# Session 4 — Make It Better, Then Break It
### Brain MRI Tumour Segmentation · CS Academy Seminar

---

Two halves today.

**First half:** honest engineering. Augmentation, threshold tuning, a bigger model. Push your Dice up.

**Second half:** we look closely at how you have been measuring, and everything you have written
down for the past two sessions stops being true.

⏱ Roughly 2 hours. GPU required.

In [ ]:
#@title Setup — run this first  { display-mode: "form" }
# Downloads the seminar helper code and the dataset.
REPO_RAW = "https://raw.githubusercontent.com/OTMAN-REPO/brain-mri-seminar/main"  #@param {type:"string"}
DATA_URL = ""  #@param {type:"string"}

import os, urllib.request
if not os.path.exists("seminar.py"):
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/seminar.py", "seminar.py")
        print("Got seminar.py")
    except Exception as e:
        raise SystemExit(f"Could not fetch seminar.py from {REPO_RAW}\n"
                         f"Upload it manually to this Colab session (folder icon on the left).\n{e}")

from seminar import *
import numpy as np, matplotlib.pyplot as plt
images, masks, patient_ids, slice_index = get_data(url=DATA_URL)
print(f"\n{len(images)} slices | {len(np.unique(patient_ids))} patients | image {images.shape[1:]}")
print("device:", DEVICE)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, time
assert DEVICE == "cuda", "Runtime -> Change runtime type -> T4 GPU"

def train_model(train_idx, val_idx, epochs=12, f=16, bs=16, augment=True, lr=1e-3,
                seed=0, verbose=True):
    """Train a U-Net and return (model, val_dice)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = UNet(f=f).to(DEVICE)
    dl = torch.utils.data.DataLoader(
        SliceDataset(images, masks, train_idx, augment=augment),
        batch_size=bs, shuffle=True, num_workers=2, drop_last=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for ep in range(epochs):
        model.train(); tot = n = 0; t0 = time.time()
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = combo_loss(model(x), y)
            loss.backward(); opt.step(); tot += loss.item(); n += 1
        sched.step()
        if verbose:
            vd = dice_score(predict_all(model, images, val_idx), masks[val_idx])
            print(f"  epoch {ep+1:2d}  loss {tot/n:.4f}  val Dice {vd:.4f}  ({time.time()-t0:.0f}s)")
    return model, dice_score(predict_all(model, images, val_idx), masks[val_idx])

train_idx, val_idx = split_by_slice(len(images), val_frac=0.2, seed=0)   # same split as Session 3
print(f"{len(train_idx)} train / {len(val_idx)} val slices")

## Part 1 — Improvement 1: data augmentation

You have a few thousand slices. That is not many. But a brain flipped left-to-right is still a brain,
and the tumour is still in the same place relative to the anatomy — so you can manufacture more
training examples for free.

The `SliceDataset` already supports `augment=True`: random horizontal flip, vertical flip, and
90° rotations. **The mask gets the identical transform** — that part is essential and easy to get
wrong.

In [ ]:
ds = SliceDataset(images, masks, train_idx, augment=True)
fig, ax = plt.subplots(2, 6, figsize=(14, 4.6))
for k in range(6):
    x, y = ds[0]                                   # same slice, six times
    ax[0,k].imshow(x[1], cmap="gray"); ax[0,k].axis("off")
    ax[1,k].imshow(y[0], cmap="gray"); ax[1,k].axis("off")
ax[0,0].set_ylabel("FLAIR"); ax[1,0].set_ylabel("mask")
plt.suptitle("one slice, six random augmentations — check the mask always matches")
plt.tight_layout(); plt.show()

In [ ]:
print("WITHOUT augmentation:")
_,      d_noaug = train_model(train_idx, val_idx, epochs=12, augment=False, verbose=False)
print(f"  val Dice {d_noaug:.4f}\n")
print("WITH augmentation:")
model,  d_aug   = train_model(train_idx, val_idx, epochs=12, augment=True, verbose=False)
print(f"  val Dice {d_aug:.4f}\n")
print(f"difference: {d_aug - d_noaug:+.4f}")

## Improvement 2: stop assuming 0.5

Your model outputs a probability per pixel, and you have been calling anything above **0.5** a
tumour. That 0.5 is a habit, not a law. When the positive class is rare, the optimum is often lower.

### Q1. Find the best threshold.

In [ ]:
import torch as T
with T.no_grad():
    probs = []
    for k in range(0, len(val_idx), 32):
        c = val_idx[k:k+32]
        x = T.from_numpy(images[c].astype(np.float32).transpose(0,3,1,2)/255.).to(DEVICE)
        probs.append(T.sigmoid(model(x)).cpu().numpy()[:,0])
    probs = np.concatenate(probs)

ths = np.arange(0.05, 0.96, 0.05)
ds_ = [dice_score(probs > t, masks[val_idx]) for t in ths]

plt.figure(figsize=(6.5,3))
plt.plot(ths, ds_, "o-", ms=4); plt.axvline(0.5, ls="--", c="grey", label="the default")
plt.xlabel("threshold"); plt.ylabel("val Dice"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

best_t = ths[int(np.argmax(ds_))]
print(f"default 0.50 -> Dice {dice_score(probs > 0.5, masks[val_idx]):.4f}")
print(f"best {best_t:.2f} -> Dice {max(ds_):.4f}")

## Improvement 3: your own experiments

### Q2. Pick two and run them. Record what happens — including if it makes things worse.

- **Wider network** — `f=32` instead of 16 (4x the parameters, slower)
- **Longer training** — 25 or 40 epochs
- **Learning rate** — try `3e-4` or `3e-3`
- **Remove small blobs** from the prediction as post-processing
- **Drop the empty slices** from training and see what it does
- **Only use the FLAIR channel** — do the other two earn their keep?

In [ ]:
# TODO: your experiments. Keep a table.
results = {"baseline (aug, thr 0.5)": d_aug}

# example:
# _, d = train_model(train_idx, val_idx, epochs=12, f=32, verbose=False)
# results["wider f=32"] = d

for k, v in results.items():
    print(f"  {k:32s} {v:.4f}")

---
---

# Part 2 — Now a question about that validation set

Everyone has a number they are happy with. Before you write it down anywhere permanent, let's check
one thing about how the split was made.

Session 3 built the split like this:

```python
train_idx, val_idx = split_by_slice(len(images), val_frac=0.2, seed=0)
```

It shuffled **all the slices together** and took 20% for validation.

### Q3. Ask a very simple question about that.

How many patients have some of their slices in training and some in validation?

In [ ]:
train_patients = set(patient_ids[train_idx])
val_patients   = set(patient_ids[val_idx])
overlap        = train_patients & val_patients

print(f"patients in training   : {len(train_patients)}")
print(f"patients in validation : {len(val_patients)}")
print(f"patients in BOTH       : {len(overlap)}")
print(f"\n{len(overlap)/len(val_patients):.0%} of the validation patients were also trained on.")

### Q4. Why is that bad? Look at the evidence.

Slices from one patient are consecutive cross-sections of the *same brain*, millimetres apart.
Slice 12 and slice 13 are nearly the same image, with nearly the same tumour in nearly the same place.

Below: pick a patient, and look at slices that ended up on opposite sides of the split.

In [ ]:
p = sorted(overlap)[0] if overlap else sorted(train_patients)[0]
tr_s = sorted(set(np.where(patient_ids == p)[0]) & set(train_idx.tolist()))
va_s = sorted(set(np.where(patient_ids == p)[0]) & set(val_idx.tolist()))

pairs = [(a, b) for a in tr_s for b in va_s if abs(slice_index[a] - slice_index[b]) == 1][:4]
if pairs:
    fig, ax = plt.subplots(2, len(pairs), figsize=(3.1*len(pairs), 6), squeeze=False)
    for k, (a, b) in enumerate(pairs):
        ax[0,k].imshow(images[a][...,1], cmap="gray")
        ax[0,k].contour(masks[a], levels=[.5], colors="lime", linewidths=1)
        ax[0,k].set_title(f"TRAIN slice {slice_index[a]}", fontsize=9, color="darkgreen")
        ax[1,k].imshow(images[b][...,1], cmap="gray")
        ax[1,k].contour(masks[b], levels=[.5], colors="red", linewidths=1)
        ax[1,k].set_title(f"VAL slice {slice_index[b]}", fontsize=9, color="darkred")
    for a_ in ax.ravel(): a_.axis("off")
    plt.suptitle(f"patient {p} — the model trained on the top row,\nthen we 'tested' it on the bottom row", y=1.02)
    plt.tight_layout(); plt.show()
print("These are adjacent cross-sections of the same brain, taken seconds apart.")

The model was not being tested on unseen data. It was being tested on **near-copies of its own
training set**. It could score well by memorising rather than generalising, and we would never know.

This is called **data leakage**. In medical imaging it is the number one cause of results that look
brilliant in a paper and collapse in a hospital.

### The honest split: by patient

A patient goes **entirely** into training or **entirely** into validation. Never both.

In [ ]:
tr_p, va_p = split_by_patient(patient_ids, val_frac=0.2, seed=0)
print(f"train {len(tr_p)} slices / {len(np.unique(patient_ids[tr_p]))} patients")
print(f"val   {len(va_p)} slices / {len(np.unique(patient_ids[va_p]))} patients")
print(f"patients in both: {len(set(patient_ids[tr_p]) & set(patient_ids[va_p]))}")

### Q5. Retrain both ways and compare.

One run proves nothing — the difference could be luck. So we run **three seeds each** and look at the
spread. This is what "is the effect real?" actually means in practice.

⏱ Six training runs. Grab a drink.

In [ ]:
rows = []
for seed in range(3):
    ts, vs = split_by_slice(len(images), val_frac=0.2, seed=seed)
    tp, vp = split_by_patient(patient_ids, val_frac=0.2, seed=seed)
    _, d_leaky  = train_model(ts, vs, epochs=12, seed=seed, verbose=False)
    _, d_honest = train_model(tp, vp, epochs=12, seed=seed, verbose=False)
    rows.append((seed, d_leaky, d_honest))
    print(f"seed {seed}:  slice-split {d_leaky:.4f}   patient-split {d_honest:.4f}   "
          f"gap {d_leaky-d_honest:+.4f}")

r = np.array(rows)
print(f"\nslice split  (leaky)  : {r[:,1].mean():.4f} ± {r[:,1].std():.4f}")
print(f"patient split (honest): {r[:,2].mean():.4f} ± {r[:,2].std():.4f}")
print(f"average inflation     : {(r[:,1]-r[:,2]).mean():+.4f}")

In [ ]:
plt.figure(figsize=(5.5,3.4))
plt.bar([0,1], [r[:,1].mean(), r[:,2].mean()],
        yerr=[r[:,1].std(), r[:,2].std()], capsize=6,
        color=["indianred","seagreen"], width=.55)
plt.xticks([0,1], ["split by slice\n(leaky)", "split by patient\n(honest)"])
plt.ylabel("validation Dice"); plt.grid(axis="y", alpha=.3)
plt.title("the same model, measured two ways"); plt.tight_layout(); plt.show()

## A confession

The Session 3 notebook used `split_by_slice` **on purpose**. I knew it was wrong when I wrote it.

I did that because being told "always split by patient" is worth almost nothing — you would nod, and
forget it inside a month. Watching your own number deflate because you measured it wrong is worth
a great deal. You will check the split of every dataset you ever touch now.

Real researchers publish this mistake. It is in the literature. The reason it survives peer review
is that the leaky number is the *better-looking* number, and nobody enjoys interrogating good news.

**The number on the right is your real score.** Use it from now on.

## Part 3 — Where does it actually fail?

Now that you can trust your validation set, look at *which patients* the model struggles on.

In [ ]:
model, _ = train_model(tr_p, va_p, epochs=20, seed=0, verbose=False)
pred = predict_all(model, images, va_p)
pp = per_patient_dice(pred, masks[va_p], patient_ids[va_p])

order = sorted(pp.items(), key=lambda kv: kv[1])
print("worst 5 patients:")
for k, v in order[:5]:  print(f"  {k}  {v:.3f}")
print("best 5 patients:")
for k, v in order[-5:]: print(f"  {k}  {v:.3f}")

plt.figure(figsize=(7,3))
plt.bar(range(len(order)), [v for _, v in order], color="steelblue")
plt.axhline(0.84, ls="--", c="crimson", label="human inter-rater agreement (~0.84)")
plt.ylabel("Dice"); plt.xlabel("validation patients, sorted"); plt.legend(fontsize=8)
plt.grid(axis="y", alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# TODO: look at the worst patient's slices. What is going on? Small tumour?
#       Faint edges? Something bright that isn't a tumour?
worst = order[0][0]
sel = np.where(patient_ids[va_p] == worst)[0]
sel = sel[np.argsort(masks[va_p][sel].reshape(len(sel),-1).sum(1))[-6:]]

fig, ax = plt.subplots(1, len(sel), figsize=(2.6*len(sel), 3))
for k, s in enumerate(sel):
    j = va_p[s]
    ax[k].imshow(images[j][...,1], cmap="gray")
    ax[k].contour(masks[j], levels=[.5], colors="lime", linewidths=1.2)
    if pred[s].any(): ax[k].contour(pred[s], levels=[.5], colors="red", linewidths=1.2)
    ax[k].set_title(f"dice {dice_score(pred[s], masks[j]):.2f}", fontsize=9); ax[k].axis("off")
plt.suptitle(f"worst patient: {worst}   (green = radiologist, red = model)")
plt.tight_layout(); plt.show()

---

## Exit ticket

1. Your **patient-split** Dice — the honest one.
2. How much was your old number inflated by?
3. In your own words, why does splitting by slice leak?
4. Your model's Dice against the ~0.84 humans get. What would you tell a hospital?
5. Describe the worst patient. Why is it hard?

Next session you choose a research question and run it yourself.

In [ ]:
#@markdown ### Session 4 exit ticket
honest_dice = ""  #@param {type:"string"}
inflation = ""  #@param {type:"string"}
why_leakage = ""  #@param {type:"string"}
what_i_would_tell_a_hospital = ""  #@param {type:"string"}
worst_patient_notes = ""  #@param {type:"string"}
print("Saved.")